In [21]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [22]:
df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")

In [23]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [24]:
df.drop(columns=['id', 'Unnamed: 32'], inplace = True)

In [25]:
print(df.shape)
df.head()

(569, 31)


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [26]:
xtrain, xtest, ytrain, ytest = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

##### Scaling

In [27]:
scalar = StandardScaler()
xtrain = scalar.fit_transform(xtrain)
xtest = scalar.transform(xtest)

##### Label Encoding

In [28]:
encoder = LabelEncoder()
ytrain = encoder.fit_transform(ytrain)
ytest = encoder.transform(ytest)

#### Numpy Array to Pytorch Tensors

In [29]:
xtrain = torch.from_numpy(xtrain)
xtest = torch.from_numpy(xtest)
ytrain = torch.from_numpy(ytrain)
ytest = torch.from_numpy(ytest)

xtrain = xtrain.float()
xtest = xtest.float()
ytrain = ytrain.float()
ytest = ytest.float()

In [30]:
print(f"X_train shape: {xtrain.shape}")
print(f"X_test shape: {xtest.shape}")
print(f"Y_train shape: {ytrain.shape}")
print(f"Y_test shape: {ytest.shape}")

X_train shape: torch.Size([455, 30])
X_test shape: torch.Size([114, 30])
Y_train shape: torch.Size([455])
Y_test shape: torch.Size([114])


#### Using Dataset and DataLoader class

In [31]:
class CustomDataset(Dataset):
    
    def __init__(self, features: torch.Tensor, labels: torch.Tensor):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

#### Create train and test dataset based on `Dataset` and `DataLoader` class

In [32]:
train_dataset = CustomDataset(xtrain, ytrain)
test_dataset = CustomDataset(xtest, ytest)

In [33]:
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=True)

#### Defining the Neural Netwwork

In [34]:
class MySimpleNeuralNetwork(nn.Module):
    
    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out = self.linear(features)
        out = self.sigmoid(out)
        return out

#### Important Parameters

In [35]:
learning_rate = 0.1
epochs = 25

#### Define Loss Function

In [36]:
loss_func = nn.BCELoss()

#### Training Pipeline

In [37]:
# create model
model = MySimpleNeuralNetwork(num_features=xtrain.shape[1])

#define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
    loss = torch.tensor(0.0)
    for batch_features, batch_labels in train_loader:

        #forward pass
        y_pred = model(batch_features)

        # loss calculation
        loss = loss_func(y_pred, batch_labels.view(-1, 1))

        # clear gradients
        optimizer.zero_grad()

        # backward pass
        loss.backward()

        # update parameters
        optimizer.step()

    print(f"epoch: {epoch+1}, Loss: {loss.item()}")

epoch: 1, Loss: 0.28597334027290344
epoch: 2, Loss: 0.2493162453174591
epoch: 3, Loss: 0.1457698494195938
epoch: 4, Loss: 0.08000823110342026
epoch: 5, Loss: 0.08895596116781235
epoch: 6, Loss: 0.1632455438375473
epoch: 7, Loss: 0.11147040128707886
epoch: 8, Loss: 0.2098540961742401
epoch: 9, Loss: 0.023944610729813576
epoch: 10, Loss: 0.026235714554786682
epoch: 11, Loss: 0.0648563802242279
epoch: 12, Loss: 0.01738373003900051
epoch: 13, Loss: 0.01018708199262619
epoch: 14, Loss: 0.08110722154378891
epoch: 15, Loss: 0.05399877950549126
epoch: 16, Loss: 0.0008594747632741928
epoch: 17, Loss: 0.014707103371620178
epoch: 18, Loss: 0.012191245332360268
epoch: 19, Loss: 0.03402629867196083
epoch: 20, Loss: 0.10033901780843735
epoch: 21, Loss: 0.010738967917859554
epoch: 22, Loss: 0.04890189692378044
epoch: 23, Loss: 0.021375317126512527
epoch: 24, Loss: 0.013389954343438148
epoch: 25, Loss: 0.016012703999876976


In [38]:
print(model.linear.weight)
print("\n")
print(model.linear.bias)

Parameter containing:
tensor([[ 0.3592,  0.5787,  0.3400,  0.6738,  0.1147,  0.1567,  0.4103,  0.7474,
          0.1315, -0.2329,  0.5478, -0.0129,  0.2291,  0.4798,  0.0607, -0.2398,
         -0.0458,  0.0737, -0.2062, -0.2396,  0.6994,  0.7054,  0.5839,  0.6469,
          0.4581,  0.1081,  0.4361,  0.7781,  0.6001, -0.0535]],
       requires_grad=True)


Parameter containing:
tensor([-0.3867], requires_grad=True)


#### Model Evaluation

In [39]:
model.eval()
accuracy_list = []

for batch_features, batch_labels in test_loader:
    y_pred = model(batch_features)
    y_pred = (y_pred >0.8).float()

    batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
    accuracy_list.append(batch_accuracy)

    overall_accuracy = sum(accuracy_list) / len(accuracy_list)

    print(f"Accuracy: {overall_accuracy:.4f}")

Accuracy: 0.9688
Accuracy: 0.9688
Accuracy: 0.9688
Accuracy: 0.9627


In [40]:
type(loss)

torch.Tensor